In [ ]:
import torch
from math import sqrt

def generate_prior_boxes():
    feature_maps = [38, 19, 10, 5, 3, 1]
    steps = [8, 16, 32, 64, 100, 300]
    min_sizes = [30, 60, 111, 162, 213, 264]
    max_sizes = [60, 111, 162, 213, 264, 315]
    aspect_ratios = [
        [2],             # Conv4_3: 4 priors
        [2, 3],          # Conv7: 6 priors
        [2, 3],          # Conv8_2: 6 priors
        [2, 3],          # Conv9_2: 6 priors
        [2],             # Conv10_2: 4 priors
        [2]              # Conv11_2: 4 priors
    ]

    priors = []
    for k, f in enumerate(feature_maps):
        for i in range(f):
            for j in range(f):
                cx = (j + 0.5) / f
                cy = (i + 0.5) / f

                s_k = min_sizes[k] / 300.0
                priors.append([cx, cy, s_k, s_k])

                s_k_prime = sqrt(s_k * (max_sizes[k] / 300.0))
                priors.append([cx, cy, s_k_prime, s_k_prime])

                for ar in aspect_ratios[k]:
                    priors.append([cx, cy, s_k * sqrt(ar), s_k / sqrt(ar)])
                    priors.append([cx, cy, s_k / sqrt(ar), s_k * sqrt(ar)])

    priors = torch.tensor(priors, dtype=torch.float32)
    priors.clamp_(min=0.0, max=1.0)
    return priors  

def point_form(boxes):
    return torch.cat([boxes[:, :2] - boxes[:, 2:] / 2,
                      boxes[:, :2] + boxes[:, 2:] / 2], 1)

def center_form(boxes):
    return torch.cat([(boxes[:, 2:] + boxes[:, :2]) / 2,
                      boxes[:, 2:] - boxes[:, :2]], 1)

def jaccard_overlap(box_a, box_b):
    A = box_a.size(0)
    B = box_b.size(0)

    max_xy = torch.min(box_a[:, 2:].unsqueeze(1).expand(A, B, 2),
                       box_b[:, 2:].unsqueeze(0).expand(A, B, 2))
    min_xy = torch.max(box_a[:, :2].unsqueeze(1).expand(A, B, 2),
                       box_b[:, :2].unsqueeze(0).expand(A, B, 2))

    inter = torch.clamp((max_xy - min_xy), min=0)
    inter_area = inter[:, :, 0] * inter[:, :, 1]

    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1])).unsqueeze(1).expand(A, B)
    area_b = ((box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1])).unsqueeze(0).expand(A, B)

    union = area_a + area_b - inter_area
    return inter_area / union  

def match(threshold, truths, priors, variances, labels, loc_t, conf_t, idx):
    overlaps = jaccard_overlap(truths, point_form(priors))

    best_prior_overlap, best_prior_idx = overlaps.max(1, keepdim=True)
    best_truth_overlap, best_truth_idx = overlaps.max(0, keepdim=True)
    best_truth_idx.squeeze_(0)
    best_truth_overlap.squeeze_(0)
    best_prior_idx.squeeze_(1)
    best_prior_overlap.squeeze_(1)

    best_truth_overlap.index_fill_(0, best_prior_idx, 2.0)
    for j in range(best_prior_idx.size(0)):
        best_truth_idx[best_prior_idx[j]] = j

    matches = truths[best_truth_idx]  
    conf = labels[best_truth_idx] + 1  
    conf[best_truth_overlap < threshold] = 0 

    g_cxcy = (matches[:, :2] + matches[:, 2:]) / 2 - priors[:, :2]
    g_cxcy /= (variances[0] * priors[:, 2:])
    g_wh = (matches[:, 2:] - matches[:, :2]) / priors[:, 2:]
    g_wh = torch.log(torch.clamp(g_wh, min=1e-10)) / variances[1]
    loc = torch.cat([g_cxcy, g_wh], 1)

    loc_t[idx] = loc
    conf_t[idx] = conf

def decode(loc, priors, variances=[0.1, 0.2]):
    boxes = torch.cat([
        priors[:, :2] + loc[:, :2] * variances[0] * priors[:, 2:],
        priors[:, 2:] * torch.exp(loc[:, 2:] * variances[1])
    ], 1)
    boxes = point_form(boxes)
    return boxes.clamp(0.0, 1.0)

def nms(boxes, scores, overlap=0.45, top_k=200):
    keep = []
    if boxes.numel() == 0:
        return torch.LongTensor(keep)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    area = (x2 - x1) * (y2 - y1)
    v, idx = scores.sort(0)
    idx = idx[-top_k:]

    while idx.numel() > 0:
        i = idx[-1].item()
        keep.append(i)
        if idx.size(0) == 1:
            break
        idx = idx[:-1]

        xx1 = torch.clamp(x1[idx], min=x1[i])
        yy1 = torch.clamp(y1[idx], min=y1[i])
        xx2 = torch.clamp(x2[idx], max=x2[i])
        yy2 = torch.clamp(y2[idx], max=y2[i])

        w = torch.clamp(xx2 - xx1, min=0.0)
        h = torch.clamp(yy2 - yy1, min=0.0)
        inter = w * h

        rem_areas = area[idx]
        union = rem_areas + area[i] - inter
        iou = inter / union

        idx = idx[iou <= overlap]
    return torch.tensor(keep, dtype=torch.long)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.model_zoo as model_zoo

class L2Norm(nn.Module):
    def __init__(self, n_channels=512, scale=20.0):
        super(L2Norm, self).__init__()
        self.n_channels = n_channels
        self.gamma = scale
        self.eps = 1e-10
        self.weight = nn.Parameter(torch.Tensor(self.n_channels))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.constant_(self.weight, self.gamma)

    def forward(self, x):
        norm = x.pow(2).sum(dim=1, keepdim=True).sqrt() + self.eps
        x = torch.div(x, norm)
        out = self.weight.unsqueeze(0).unsqueeze(2).unsqueeze(3).expand_as(x) * x
        return out


class VGG16Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(VGG16Backbone, self).__init__()

        # Блок 1 (300x300 -> 150x150)
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Блок 2 (150x150 -> 75x75)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Блок 3 (75x75 -> 38x38 завдяки ceil_mode=True)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

        # Блок 4 (вихід для Map 1: Conv4_3)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True) # 38x38 -> 19x19

        # Блок 5 (19x19 -> 19x19)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)

        # FC6 -> Conv6 (Dilated/Atrous згортка з dilation=6)
        self.conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
        # FC7 -> Conv7 (вихід для Map 2: Conv7)
        self.conv7 = nn.Conv2d(1024, 1024, kernel_size=1)

        self._init_weights()
        if pretrained:
            self._load_pretrained_vgg()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def _load_pretrained_vgg(self):
        vgg16_url = 'https://download.pytorch.org/models/vgg16-397923af.pth'
        state_dict = model_zoo.load_url(vgg16_url, progress=False)

        vgg_layers = [
            self.conv1_1, self.conv1_2,
            self.conv2_1, self.conv2_2,
            self.conv3_1, self.conv3_2, self.conv3_3,
            self.conv4_1, self.conv4_2, self.conv4_3,
            self.conv5_1, self.conv5_2, self.conv5_3
        ]

        pt_keys = [k for k in state_dict.keys() if 'features' in k and 'weight' in k]
        bias_keys = [k for k in state_dict.keys() if 'features' in k and 'bias' in k]

        for i, layer in enumerate(vgg_layers):
            layer.weight.data.copy_(state_dict[pt_keys[i]])
            layer.bias.data.copy_(state_dict[bias_keys[i]])

    def forward(self, x):
        x = F.relu(self.conv1_1(x), inplace=True)
        x = F.relu(self.conv1_2(x), inplace=True)
        x = self.pool1(x)

        x = F.relu(self.conv2_1(x), inplace=True)
        x = F.relu(self.conv2_2(x), inplace=True)
        x = self.pool2(x)

        x = F.relu(self.conv3_1(x), inplace=True)
        x = F.relu(self.conv3_2(x), inplace=True)
        x = F.relu(self.conv3_3(x), inplace=True)
        x = self.pool3(x)

        x = F.relu(self.conv4_1(x), inplace=True)
        x = F.relu(self.conv4_2(x), inplace=True)
        conv4_3_out = F.relu(self.conv4_3(x), inplace=True) # Map 1 (38x38)

        x = self.pool4(conv4_3_out)
        x = F.relu(self.conv5_1(x), inplace=True)
        x = F.relu(self.conv5_2(x), inplace=True)
        x = F.relu(self.conv5_3(x), inplace=True)
        x = self.pool5(x)

        x = F.relu(self.conv6(x), inplace=True)
        conv7_out = F.relu(self.conv7(x), inplace=True)     

        return conv4_3_out, conv7_out


class SSD300(nn.Module):
    def __init__(self, num_classes=21, pretrained_vgg=True):
        super(SSD300, self).__init__()
        self.num_classes = num_classes

        self.backbone = VGG16Backbone(pretrained=pretrained_vgg)
        self.l2_norm = L2Norm(512, scale=20.0)

        # Map 3: 10x10
        self.conv8_1 = nn.Conv2d(1024, 256, kernel_size=1)
        self.conv8_2 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)

        # Map 4: 5x5
        self.conv9_1 = nn.Conv2d(512, 128, kernel_size=1)
        self.conv9_2 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)

        # Map 5: 3x3
        self.conv10_1 = nn.Conv2d(256, 128, kernel_size=1)
        self.conv10_2 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=0)

        # Map 6: 1x1
        self.conv11_1 = nn.Conv2d(256, 128, kernel_size=1)
        self.conv11_2 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=0)

        self.loc_heads = nn.ModuleList([
            nn.Conv2d(512, 4 * 4, kernel_size=3, padding=1),
            nn.Conv2d(1024, 6 * 4, kernel_size=3, padding=1),
            nn.Conv2d(512, 6 * 4, kernel_size=3, padding=1),
            nn.Conv2d(256, 6 * 4, kernel_size=3, padding=1),
            nn.Conv2d(256, 4 * 4, kernel_size=3, padding=1),
            nn.Conv2d(256, 4 * 4, kernel_size=3, padding=1)
        ])

        self.conf_heads = nn.ModuleList([
            nn.Conv2d(512, 4 * self.num_classes, kernel_size=3, padding=1),
            nn.Conv2d(1024, 6 * self.num_classes, kernel_size=3, padding=1),
            nn.Conv2d(512, 6 * self.num_classes, kernel_size=3, padding=1),
            nn.Conv2d(256, 6 * self.num_classes, kernel_size=3, padding=1),
            nn.Conv2d(256, 4 * self.num_classes, kernel_size=3, padding=1),
            nn.Conv2d(256, 4 * self.num_classes, kernel_size=3, padding=1)
        ])

        self._init_extra_weights()

    def _init_extra_weights(self):
        for m in [self.conv8_1, self.conv8_2, self.conv9_1, self.conv9_2,
                  self.conv10_1, self.conv10_2, self.conv11_1, self.conv11_2]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        sources = []
        loc = []
        conf = []

        conv4_3, conv7 = self.backbone(x)
        sources.append(self.l2_norm(conv4_3))

        sources.append(conv7)
        
        x = F.relu(self.conv8_1(conv7), inplace=True)
        x = F.relu(self.conv8_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv9_1(x), inplace=True)
        x = F.relu(self.conv9_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv10_1(x), inplace=True)
        x = F.relu(self.conv10_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv11_1(x), inplace=True)
        x = F.relu(self.conv11_2(x), inplace=True)
        sources.append(x)

        for (feat, l_head, c_head) in zip(sources, self.loc_heads, self.conf_heads):
            loc.append(l_head(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4))
            conf.append(c_head(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.num_classes))

        loc = torch.cat(loc, 1)   
        conf = torch.cat(conf, 1) 

        return loc, conf

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiBoxLoss(nn.Module):
    def __init__(self, num_classes=21, overlap_thresh=0.5, neg_pos_ratio=3):
        super(MultiBoxLoss, self).__init__()
        self.num_classes = num_classes
        self.threshold = overlap_thresh
        self.negpos_ratio = neg_pos_ratio
        self.variances = [0.1, 0.2]

    def forward(self, predictions, priors, targets):
        loc_data, conf_data = predictions
        num = loc_data.size(0)
        num_priors = priors.size(0)

        loc_t = torch.zeros(num, num_priors, 4, dtype=torch.float32, device=loc_data.device)
        conf_t = torch.zeros(num, num_priors, dtype=torch.long, device=loc_data.device)

        for idx in range(num):
            truths = targets[idx][:, :-1].to(loc_data.device)
            labels = targets[idx][:, -1].to(loc_data.device).long()
            match(self.threshold, truths, priors, self.variances, labels, loc_t, conf_t, idx)

        pos = conf_t > 0    
        num_pos = pos.sum(dim=1, keepdim=True)

        pos_idx = pos.unsqueeze(pos.dim()).expand_as(loc_data)
        loc_p = loc_data[pos_idx].view(-1, 4)
        loc_target = loc_t[pos_idx].view(-1, 4)
        loss_l = F.smooth_l1_loss(loc_p, loc_target, reduction='sum')


        batch_conf = conf_data.view(-1, self.num_classes)
        loss_c = F.cross_entropy(batch_conf, conf_t.view(-1), reduction='none')
        loss_c = loss_c.view(num, -1)

        loss_c[pos] = 0.0

        _, loss_idx = loss_c.sort(1, descending=True)
        _, idx_rank = loss_idx.sort(1)

        num_pos = pos.long().sum(1, keepdim=True)
        num_neg = torch.clamp(self.negpos_ratio * num_pos, max=pos.size(1) - 1)
        neg = idx_rank < num_neg.expand_as(idx_rank)


        pos_idx_c = pos.unsqueeze(2).expand_as(conf_data)
        neg_idx_c = neg.unsqueeze(2).expand_as(conf_data)

        conf_p = conf_data[(pos_idx_c + neg_idx_c).gt(0)].view(-1, self.num_classes)
        targets_weighted = conf_t[(pos + neg).gt(0)]
        loss_c = F.cross_entropy(conf_p, targets_weighted, reduction='sum')

        N = num_pos.data.sum().float()
        N = max(N, 1.0)
        return loss_l / N, loss_c / N

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms.functional as FT


VOC_CLASSES = (
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
)
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES) + 1  


class PascalVOCDataset(Dataset):
    def __init__(self, root='./data', year='2007', image_set='trainval', download=True):
        self.voc = torchvision.datasets.VOCDetection(
            root=root, year=year, image_set=image_set, download=download
        )
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w_orig, h_orig = image.size

        boxes = []
        labels = []

        objects = target['annotation']['object']
        if not isinstance(objects, list):
            objects = [objects]

        for obj in objects:
            name = obj['name']
            if name not in CLASS_TO_IDX:
                continue
            labels.append(CLASS_TO_IDX[name])

            bndbox = obj['bndbox']
            xmin = float(bndbox['xmin']) / w_orig
            ymin = float(bndbox['ymin']) / h_orig
            xmax = float(bndbox['xmax']) / w_orig
            ymax = float(bndbox['ymax']) / h_orig
            boxes.append([xmin, ymin, xmax, ymax])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.long)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.long)

        image = image.resize((300, 300), Image.BILINEAR)
        image = FT.to_tensor(image)
        image = FT.normalize(image, mean=self.mean, std=self.std)

        targets = torch.cat([boxes, labels.unsqueeze(1).float()], dim=1)
        return image, targets


def detection_collate(batch):
    images = [sample[0] for sample in batch]
    targets = [sample[1] for sample in batch]
    return torch.stack(images, 0), targets


def train_ssd(num_epochs=10, batch_size=8, lr=1e-3, data_root='./data'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Тренування запущено на пристрої: {device}")

    train_dataset = PascalVOCDataset(root=data_root, year='2007', image_set='trainval', download=True)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=detection_collate,
        num_workers=2,
        pin_memory=True if device.type == 'cuda' else False
    )


    model = SSD300(num_classes=NUM_CLASSES, pretrained_vgg=True).to(device)
    priors = generate_prior_boxes().to(device)
    criterion = MultiBoxLoss(num_classes=NUM_CLASSES, overlap_thresh=0.5, neg_pos_ratio=3)

    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[6, 8], gamma=0.1)

    loc_history, conf_history, total_history = [], [], []

    for epoch in range(num_epochs):
        model.train()
        running_loc, running_conf, running_total = 0.0, 0.0, 0.0

        for i, (images, targets) in enumerate(train_loader):
            images = images.to(device)

            optimizer.zero_grad()
            loc_preds, conf_preds = model(images)

            loss_l, loss_c = criterion((loc_preds, conf_preds), priors, targets)
            loss = loss_l + loss_c

            loss.backward()
            optimizer.step()

            running_loc += loss_l.item()
            running_conf += loss_c.item()
            running_total += loss.item()

            if (i + 1) % 50 == 0 or (i + 1) == len(train_loader):
                print(f"Епоха [{epoch+1}/{num_epochs}], Крок [{i+1}/{len(train_loader)}] | "
                      f"Loc Loss: {running_loc/(i+1):.4f} | "
                      f"Conf Loss: {running_conf/(i+1):.4f} | "
                      f"Total: {running_total/(i+1):.4f}")

        scheduler.step()
        loc_history.append(running_loc / len(train_loader))
        conf_history.append(running_conf / len(train_loader))
        total_history.append(running_total / len(train_loader))

    os.makedirs('./weights', exist_ok=True)
    weights_path = './weights/ssd300_voc.pth'
    torch.save(model.state_dict(), weights_path)
    print(f"Ваги збережено до {weights_path}")

    plt.figure(figsize=(9, 4.5))
    plt.plot(total_history, label='Total MultiBox Loss', color='black', linewidth=2)
    plt.plot(loc_history, label='Localization Loss (Smooth L1)', color='crimson', linestyle='--')
    plt.plot(conf_history, label='Confidence Loss (Cross-Entropy)', color='royalblue', linestyle='--')
    plt.xlabel('Епоха')
    plt.ylabel('Loss')
    plt.title('Динаміка збіжності функції втрат SSD300 (Pascal VOC 2007)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('ssd_training_loss.png', dpi=300, bbox_inches='tight')
    plt.show()

    return model, priors


def detect_and_draw(model, priors, image_path, conf_threshold=0.25, nms_threshold=0.45, output_file='detection_result.png'):
    device = next(model.parameters()).device
    model.eval()

    orig_img = Image.open(image_path).convert("RGB")
    w_orig, h_orig = orig_img.size

    img_tensor = orig_img.resize((300, 300), Image.BILINEAR)
    img_tensor = FT.to_tensor(img_tensor)
    img_tensor = FT.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        loc_preds, conf_preds = model(img_tensor)

    decoded_boxes = decode(loc_preds[0], priors)
    conf_scores = torch.softmax(conf_preds[0], dim=-1)

    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(orig_img)

    detected_count = 0
    for c in range(1, NUM_CLASSES):
        c_scores = conf_scores[:, c]
        mask = c_scores > conf_threshold

        if mask.sum() == 0:
            continue

        c_boxes = decoded_boxes[mask]
        c_scores = c_scores[mask]

        keep = nms(c_boxes, c_scores, overlap=nms_threshold, top_k=50)
        c_boxes = c_boxes[keep]
        c_scores = c_scores[keep]

        for box, score in zip(c_boxes, c_scores):
            detected_count += 1
            xmin = box[0].item() * w_orig
            ymin = box[1].item() * h_orig
            xmax = box[2].item() * w_orig
            ymax = box[3].item() * h_orig

            width = xmax - xmin
            height = ymax - ymin

            rect = patches.Rectangle(
                (xmin, ymin), width, height,
                linewidth=2.5, edgecolor='#00FF00', facecolor='none'
            )
            ax.add_patch(rect)

            class_name = VOC_CLASSES[c - 1]
            ax.text(
                xmin, ymin - 4, f"{class_name}: {score.item():.2f}",
                bbox=dict(facecolor='#00FF00', alpha=0.8, pad=2, edgecolor='none'),
                fontsize=10, color='black', weight='bold'
            )

    ax.axis('off')
    plt.title(f"Результат детекції SSD300 (Виявлено об'єктів: {detected_count})")
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Зображення з розміткою збережено до {output_file}")


if __name__ == '__main__':
    trained_model, priors = train_ssd(num_epochs=5, batch_size=8, lr=1e-3)
    sample_img = './data/VOCdevkit/VOC2007/JPEGImages/000005.jpg'
    if os.path.exists(sample_img):
        detect_and_draw(trained_model, priors, sample_img, conf_threshold=0.25)
    else:
        print(f"Файл {sample_img} не знайдено для візуалізації.")

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torchvision.transforms.functional as FT

VOC_CLASSES = (
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
)
NUM_CLASSES = len(VOC_CLASSES) + 1

random.seed(42)
CLASS_COLORS = [f"#{random.randint(0, 0xFFFFFF):06x}" for _ in range(NUM_CLASSES)]



def run_batch_inference(
    image_paths,
    weights_path='./weights/ssd300_voc.pth',
    conf_threshold=0.25,
    nms_threshold=0.45,
    output_grid_file='batch_detection_results.png'
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Інференс на пристрої: {device}")

    model = SSD300(num_classes=NUM_CLASSES, pretrained_vgg=False).to(device)
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Файл ваг {weights_path} не знайдено! Спочатку натренуйте модель.")
    
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()

    priors = generate_prior_boxes().to(device)

    num_images = len(image_paths)
    cols = 2
    rows = (num_images + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(14, 12))
    axes = axes.flatten() if num_images > 1 else [axes]

    for idx, img_path in enumerate(image_paths):
        ax = axes[idx]
        orig_img = Image.open(img_path).convert("RGB")
        w_orig, h_orig = orig_img.size

        img_tensor = orig_img.resize((300, 300), Image.BILINEAR)
        img_tensor = FT.to_tensor(img_tensor)
        img_tensor = FT.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        img_tensor = img_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            loc_preds, conf_preds = model(img_tensor)

        decoded_boxes = decode(loc_preds[0], priors)
        conf_scores = torch.softmax(conf_preds[0], dim=-1)

        ax.imshow(orig_img)
        detected_count = 0

        for c in range(1, NUM_CLASSES):
            c_scores = conf_scores[:, c]
            mask = c_scores > conf_threshold

            if mask.sum() == 0:
                continue

            c_boxes = decoded_boxes[mask]
            c_scores = c_scores[mask]

            keep = nms(c_boxes, c_scores, overlap=nms_threshold, top_k=20)
            c_boxes = c_boxes[keep]
            c_scores = c_scores[keep]

            for box, score in zip(c_boxes, c_scores):
                detected_count += 1
                xmin = box[0].item() * w_orig
                ymin = box[1].item() * h_orig
                xmax = box[2].item() * w_orig
                ymax = box[3].item() * h_orig

                color = CLASS_COLORS[c]

                rect = patches.Rectangle(
                    (xmin, ymin), xmax - xmin, ymax - ymin,
                    linewidth=2.5, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)

                class_name = VOC_CLASSES[c - 1]
                ax.text(
                    xmin, max(ymin - 5, 15), f"{class_name} {score.item():.2f}",
                    bbox=dict(facecolor=color, alpha=0.85, pad=2, edgecolor='none'),
                    fontsize=9, color='white', weight='bold'
                )

        img_name = os.path.basename(img_path)
        ax.set_title(f"{img_name} ({detected_count} objs)", fontsize=11)
        ax.axis('off')


    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(output_grid_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Колаж успішно згенеровано та збережено до: {output_grid_file}")


sample_paths = [
    './data/VOCdevkit/VOC2007/JPEGImages/000044.jpg',
    './data/VOCdevkit/VOC2007/JPEGImages/000046.jpg',
    './data/VOCdevkit/VOC2007/JPEGImages/000063.jpg',
    './data/VOCdevkit/VOC2007/JPEGImages/000066.jpg'
]

existing_paths = [p for p in sample_paths if os.path.exists(p)]

if existing_paths:
    run_batch_inference(
        image_paths=existing_paths,
        weights_path='./weights/ssd300_voc.pth',
        conf_threshold=0.4,
        nms_threshold=0.3,
        output_grid_file='batch_detection_results.png'
    )
else:
    print("Вказані файли не знайдено за шляхом ./data/VOCdevkit/VOC2007/JPEGImages/")